# From PyTorch to MAX

Introductory tutorial for PyTorch users who want to learn MAX's modeling APIs for **inference**. 

**API Philosophy: MAX API is PyTorch-like `nn` and Numpy-like ndarray/tensor manipulation.**

What we'll cover:

1. **Tensors** - Creating and manipulating tensors
2. **Functional API** - Common operations like relu, softmax, matmul
3. **Random Module** - Random tensor generation
4. **Module System** - Building models with `nn.Module`
5. **Real Model** - Translating a HuggingFace GPT-2 attention to MAX
6. **Compilation** - Graph compilation for optimized inference

Each section shows PyTorch code alongside the equivalent MAX code.

## Goal

Build a mental model for MAX API and a dictionary for translating PyTorch code to MAX.

## Caveats

* We're showing MAX new _experimental_ API released in 25.7
* We're working on making the API more ergonomics and fixes in the next release in 26.1

---
## 1. Tensors: PyTorch vs MAX (~5 min)

Let's start with the fundamental building block: tensors.


In [1]:
# ============ PyTorch ============
import torch

# Creating tensors
pt_tensor = torch.tensor([1.0, 2.0, 3.0])
pt_zeros = torch.zeros(3, 4)
pt_ones = torch.ones(2, 3)

print("PyTorch:")
print(f"  Tensor: {pt_tensor}")
print(f"  dtype: {pt_tensor.dtype}, shape: {pt_tensor.shape}, device: {pt_tensor.device}")

PyTorch:
  Tensor: tensor([1., 2., 3.])
  dtype: torch.float32, shape: torch.Size([3]), device: cpu


In [2]:
# ============ MAX ============
from max.experimental import Tensor

# Creating tensors
max_tensor = Tensor.constant([1.0, 2.0, 3.0])
max_zeros = Tensor.zeros([3, 4])
max_ones = Tensor.ones([2, 3])

print("MAX:")
print(f"  Tensor: {max_tensor}")
print(f"  dtype: {max_tensor.dtype}, shape: {max_tensor.shape}, device: {max_tensor.device}")

MAX:
  Tensor: TensorType(dtype=bfloat16, shape=[Dim(3)], device=gpu:0): [1.0, 2.0, 3.0]
  dtype: DType.bfloat16, shape: [Dim(3)], device: Device(type=gpu,id=0)


### Key Differences

| Concept | PyTorch | MAX |
|---------|---------|-----|
| Create from list | `torch.tensor([1,2,3])` | `Tensor.constant([1,2,3])` |
| Zeros | `torch.zeros(3, 4)` | `Tensor.zeros([3, 4])` |
| Ones | `torch.ones(2, 3)` | `Tensor.ones([2, 3])` |
| Shape syntax | `(3, 4)` tuple | `[3, 4]` list |
| Default dtype | `float32` | `bfloat16` |
| Default device | `cpu` | `gpu:0` (if available) |


In [3]:
# MAX has smart defaults - check what they are
from max.experimental.tensor import defaults

dtype, device = defaults()
print(f"MAX default dtype: {dtype}")
print(f"MAX default device: {device}")

MAX default dtype: DType.bfloat16
MAX default device: Device(type=gpu,id=0)


### Device Management


In [4]:
# ============ PyTorch Device Management ============
pt_cpu = torch.tensor([1.0, 2.0, 3.0])
if torch.cuda.is_available():
    pt_gpu = pt_cpu.to('cuda')
    pt_back = pt_gpu.to('cpu')
    print(f"PyTorch GPU tensor device: {pt_gpu.device}")
else:
    print("CUDA not available for PyTorch")

PyTorch GPU tensor device: cuda:0


In [5]:
# ============ MAX Device Management ============
from max.driver import CPU, Accelerator, accelerator_count

# Check for GPU
has_gpu = accelerator_count() > 0
print(f"GPU available: {has_gpu}")

# Create on GPU (default), move to CPU
max_gpu = Tensor.constant([1.0, 2.0, 3.0])  # On GPU by default
max_cpu = max_gpu.to(CPU())

print(f"Original device: {max_gpu.device}")
print(f"After .to(CPU()): {max_cpu.device}")

# Move back to GPU
if has_gpu:
    max_back = max_cpu.to(Accelerator())
    print(f"Back to GPU: {max_back.device}")

GPU available: True
Original device: Device(type=gpu,id=0)
After .to(CPU()): Device(type=cpu,id=0)
Back to GPU: Device(type=gpu,id=0)


### Data Types


In [6]:
# ============ PyTorch DTypes ============
pt_f32 = torch.tensor([1.0], dtype=torch.float32)
pt_f16 = pt_f32.half()  # Convert to float16
pt_bf16 = pt_f32.bfloat16()  # Convert to bfloat16
print(f"PyTorch: f32={pt_f32.dtype}, f16={pt_f16.dtype}, bf16={pt_bf16.dtype}")

PyTorch: f32=torch.float32, f16=torch.float16, bf16=torch.bfloat16


In [7]:
# ============ MAX DTypes ============
from max.dtype import DType

max_bf16 = Tensor.constant([1.0])  # Default is bfloat16
max_f32 = Tensor.constant([1.0], dtype=DType.float32)

from max.experimental.tensor import default_dtype
with default_dtype(DType.float32):
    max_another_f32 = Tensor.constant([42.])
    
max_bf16_cast_f32 = max_bf16.cast(DType.float32)
max_bf16_cast_f16 = max_bf16.cast(DType.float16)

print(
    f"MAX DTypes:\n"
    f"  bf16          = {max_bf16.dtype}\n"
    f"  f32           = {max_f32.dtype}\n"
    f"  another_f32   = {max_another_f32.dtype}\n"
    f"  bf16_cast_f32 = {max_bf16_cast_f32.dtype}\n"
    f"  bf16_cast_f16 = {max_bf16_cast_f16.dtype}")

MAX DTypes:
  bf16          = DType.bfloat16
  f32           = DType.float32
  another_f32   = DType.float32
  bf16_cast_f32 = DType.float32
  bf16_cast_f16 = DType.float16


### NumPy/PyTorch Tensor Interoperability


In [8]:
import numpy as np

# ============ PyTorch <-> NumPy ============
np_arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)
pt_from_np = torch.from_numpy(np_arr)
back_to_np = pt_from_np.numpy()
print(f"PyTorch from numpy: {pt_from_np}")

PyTorch from numpy: tensor([1., 2., 3.])


In [9]:
# ============ MAX <-> NumPy via DLPack ============
np_arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)

# NumPy to MAX (uses DLPack protocol)
max_from_np = Tensor.from_dlpack(np_arr)
print(f"MAX from numpy: {max_from_np}")

# MAX to NumPy (need to move to CPU first for bfloat16)
max_cpu = Tensor.constant([1.0, 2.0, 3.0], dtype=DType.float32, device=CPU())
back_to_np = np.from_dlpack(max_cpu)
print(f"Back to numpy: {back_to_np}")

MAX from numpy: TensorType(dtype=float32, shape=[Dim(3)], device=cpu:0): [1.0, 2.0, 3.0]
Back to numpy: [1. 2. 3.]


In [10]:
# Similarly PyTorch interop with DLPack
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
torch_tensor = torch.tensor([1., 2, 3], device=device)
print(f"torch tensor: {torch_tensor}")
max_tensor_from_torch = Tensor.from_dlpack(torch_tensor)
print(f"max tensor from torch: {max_tensor_from_torch}")

torch tensor: tensor([1., 2., 3.], device='cuda:0')
max tensor from torch: TensorType(dtype=float32, shape=[Dim(3)], device=gpu:0): [1.0, 2.0, 3.0]


---
## 2. Functional API: PyTorch vs MAX (~2 min)

Both frameworks provide functional APIs for tensor operations.


In [11]:
import torch.nn.functional as F_torch
from max.experimental import functional as F_max

### Activation Functions


In [12]:
# ============ PyTorch Activations ============
x_pt = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

print("PyTorch:")
print(f"  ReLU: {F_torch.relu(x_pt)}")
print(f"  GELU: {F_torch.gelu(x_pt)}")
print(f"  SiLU: {F_torch.silu(x_pt)}")
print(f"  Tanh: {torch.tanh(x_pt)}")

PyTorch:
  ReLU: tensor([0., 0., 0., 1., 2.])
  GELU: tensor([-0.0455, -0.1587,  0.0000,  0.8413,  1.9545])
  SiLU: tensor([-0.2384, -0.2689,  0.0000,  0.7311,  1.7616])
  Tanh: tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])


In [13]:
# ============ MAX Activations ============
x_max = Tensor.constant([-2.0, -1.0, 0.0, 1.0, 2.0])

print("MAX:")
print(f"  ReLU: {F_max.relu(x_max)}")
print(f"  GELU: {F_max.gelu(x_max)}")
print(f"  SiLU: {F_max.silu(x_max)}")
print(f"  Tanh: {F_max.tanh(x_max)}")

MAX:
  ReLU: TensorType(dtype=bfloat16, shape=[Dim(5)], device=gpu:0): [0.0, 0.0, 0.0, 1.0, 2.0]
  GELU: TensorType(dtype=bfloat16, shape=[Dim(5)], device=gpu:0): [-0.04541015625, -0.158203125, 0.0, 0.83984375, 1.953125]
  SiLU: TensorType(dtype=bfloat16, shape=[Dim(5)], device=gpu:0): [-0.23828125, -0.26953125, 0.0, 0.73046875, 1.7578125]
  Tanh: TensorType(dtype=bfloat16, shape=[Dim(5)], device=gpu:0): [-0.9609375, -0.7578125, 0.0, 0.7578125, 0.9609375]


### Softmax and LogSoftmax


In [14]:
# ============ PyTorch ============
torch_device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

logits_pt = torch.tensor([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]], dtype=torch.bfloat16, device=torch_device)
print("PyTorch:")
print(f"  Softmax: {F_torch.softmax(logits_pt, dim=-1)}")
print(f"  LogSoftmax: {F_torch.log_softmax(logits_pt, dim=-1)}")

PyTorch:
  Softmax: tensor([[0.0898, 0.2451, 0.6641],
        [0.3340, 0.3340, 0.3340]], device='cuda:0', dtype=torch.bfloat16)
  LogSoftmax: tensor([[-2.4062, -1.4062, -0.4082],
        [-1.1016, -1.1016, -1.1016]], device='cuda:0', dtype=torch.bfloat16)


In [15]:
# ============ MAX ============
logits_max = Tensor.constant([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
print("MAX:")
print(f"  Softmax: {F_max.softmax(logits_max, axis=-1)}")
print(f"  LogSoftmax: {F_max.logsoftmax(logits_max, axis=-1)}")

MAX:
  Softmax: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(3)], device=gpu:0): [0.0908203125, 0.2451171875, 0.66796875, 0.333984375, 0.333984375, 0.333984375]
  LogSoftmax: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(3)], device=gpu:0): [-2.40625, -1.40625, -0.41015625, -1.09375, -1.09375, -1.09375]


### Matrix Operations


In [16]:
# ============ PyTorch ============
A_pt = torch.randn(2, 3)
B_pt = torch.randn(3, 4)

# Matrix multiplication
C_pt = A_pt @ B_pt  # or torch.matmul(A_pt, B_pt)
print(f"PyTorch matmul shape: {C_pt.shape}")

# Transpose
print(f"PyTorch transpose: {A_pt.T.shape}")

PyTorch matmul shape: torch.Size([2, 4])
PyTorch transpose: torch.Size([3, 2])


In [17]:
# ============ MAX ============
from max.experimental import random

A_max = random.uniform([2, 3])
B_max = random.uniform([3, 4])

# Matrix multiplication
C_max = A_max @ B_max  # Same syntax!
print(f"MAX matmul shape: {C_max.shape}")

# Transpose
print(f"MAX transpose: {A_max.T.shape}")

MAX matmul shape: [Dim(2), Dim(4)]
MAX transpose: [Dim(3), Dim(2)]


In [18]:
# TODO: multi-dim tensor repr
C_max

TensorType(dtype=bfloat16, shape=[Dim(2), Dim(4)], device=gpu:0): [1.1171875, 0.96875, 0.470703125, 0.98828125, 1.515625, 1.4765625, 1.2734375, 1.2890625]

### Reduction Operations


In [19]:
# ============ PyTorch ============
x_pt = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("PyTorch:")
print(f"  Sum all: {x_pt.sum()}")
print(f"  Sum dim=1: {x_pt.sum(dim=1)}")
print(f"  Mean: {x_pt.mean()}")
print(f"  Max: {x_pt.max()}")
print(f"  Argmax dim=1: {x_pt.argmax(dim=1)}")

PyTorch:
  Sum all: 21.0
  Sum dim=1: tensor([ 6., 15.])
  Mean: 3.5
  Max: 6.0
  Argmax dim=1: tensor([2, 2])


In [20]:
# ============ MAX ============
x_max = Tensor.constant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("MAX:")
# TODO: reduction with `axis=None` i.e. over all elements
print(f"  Sum axis=1: {F_max.sum(x_max, axis=1)}")
print(f"  Mean axis=1: {F_max.mean(x_max, axis=1)}")
print(f"  Max axis=1: {F_max.max(x_max, axis=1)}")
print(f"  Argmax axis=1: {F_max.argmax(x_max, axis=1)}")

MAX:
  Sum axis=1: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(1)], device=gpu:0): [6.0, 15.0]
  Mean axis=1: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(1)], device=gpu:0): [2.0, 5.0]
  Max axis=1: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(1)], device=gpu:0): [3.0, 6.0]
  Argmax axis=1: TensorType(dtype=int64, shape=[Dim(2), Dim(1)], device=gpu:0): [2, 2]


In [21]:
# also a few most used reduction ops are available as tensor methods
print("MAX:")
print(f"  Mean axis=1: {x_max.mean(axis=1)}")
print(f"  Max axis=1: {x_max.max(axis=1)}")
print(f"  Argmax axis=1: {x_max.argmax(axis=1)}")

MAX:
  Mean axis=1: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(1)], device=gpu:0): [2.0, 5.0]
  Max axis=1: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(1)], device=gpu:0): [3.0, 6.0]
  Argmax axis=1: TensorType(dtype=int64, shape=[Dim(2), Dim(1)], device=gpu:0): [2, 2]


### Functional API Comparison Table

| Operation | PyTorch | MAX |
|-----------|---------|-----|
| ReLU | `F.relu(x)` | `F.relu(x)` |
| GELU | `F.gelu(x)` | `F.gelu(x)` |
| Softmax | `F.softmax(x, dim=-1)` | `F.softmax(x, axis=-1)` |
| MatMul | `x @ y` | `x @ y` |
| Transpose | `x.T` | `x.T` |
| Sum | `x.sum(dim=1)` | `F.sum(x, axis=1)` |
| Mean | `x.mean(dim=1)` | `F.mean(x, axis=1)` |
| Reshape | `x.view(2, 3)` | `F.reshape(x, [2, 3])` |
| Squeeze | `x.squeeze(0)` | `F.squeeze(x, axis=0)` |
| Cast | `x.float()` | `x.cast(DType.float32)` |

**Note:** MAX uses `axis` instead of `dim` for consistency with NumPy.


---
## 3. Random Module

Generating random tensors for testing and initialization.


In [22]:
# ============ PyTorch Random ============
print("PyTorch:")
print(f"  randn (normal): {torch.randn(2, 3).shape}")
print(f"  rand (uniform [0,1)): {torch.rand(2, 3).shape}")
print(f"  randint: {torch.rand(2, 3)}")

PyTorch:
  randn (normal): torch.Size([2, 3])
  rand (uniform [0,1)): torch.Size([2, 3])
  randint: tensor([[0.2017, 0.9874, 0.5723],
        [0.2468, 0.0697, 0.9628]])


In [23]:
# ============ MAX Random ============
from max.experimental import random

print("MAX:")
print(f"  normal: {random.normal([2, 3]).shape}")
print(f"  uniform: {random.uniform([2, 3]).shape}")
print(f"  uniform sample: {random.uniform([2, 3])}")

MAX:
  normal: [Dim(2), Dim(3)]
  uniform: [Dim(2), Dim(3)]
  uniform sample: TensorType(dtype=bfloat16, shape=[Dim(2), Dim(3)], device=gpu:0): [0.265625, 0.71875, 0.69921875, 0.140625, 0.470703125, 0.515625]


---
## 4. Module System: PyTorch vs MAX (~ 5min)

Building neural network models.


### Linear Layers


In [24]:
# ============ PyTorch Linear ============

pt_linear = torch.nn.Linear(in_features=10, out_features=5)
pt_input = torch.randn(2, 10)
pt_output = pt_linear(pt_input)

print(f"PyTorch Linear:")
print(f"  Weight shape: {pt_linear.weight.shape}")
print(f"  Bias shape: {pt_linear.bias.shape}")
print(f"  Output shape: {pt_output.shape}")

PyTorch Linear:
  Weight shape: torch.Size([5, 10])
  Bias shape: torch.Size([5])
  Output shape: torch.Size([2, 5])


In [25]:
# ============ MAX Linear ============
import max.nn.module_v3 as nn

max_linear = nn.Linear(in_dim=10, out_dim=5)
max_input = random.uniform([2, 10])
max_output = max_linear(max_input)

print(f"MAX Linear:")
print(f"  Weight shape: {max_linear.weight.shape}")
print(f"  Bias shape: {max_linear.bias.shape}")
print(f"  Output shape: {max_output.shape}")

MAX Linear:
  Weight shape: [Dim(5), Dim(10)]
  Bias shape: [Dim(5)]
  Output shape: [Dim(2), Dim(5)]


### Embeddings


In [26]:
# ============ PyTorch Embedding ============
pt_embed = torch.nn.Embedding(num_embeddings=1000, embedding_dim=128)
pt_tokens = torch.tensor([0, 5, 10, 100])
pt_embedded = pt_embed(pt_tokens)

print(f"PyTorch Embedding:")
print(f"  Input shape: {pt_tokens.shape}")
print(f"  Output shape: {pt_embedded.shape}")

PyTorch Embedding:
  Input shape: torch.Size([4])
  Output shape: torch.Size([4, 128])


In [27]:
# ============ MAX Embedding ============
max_embed = nn.Embedding(vocab_size=1000, dim=128)
max_tokens = Tensor.constant([0, 5, 10, 100], dtype=DType.uint64)
max_embedded = max_embed(max_tokens)

print(f"MAX Embedding:")
print(f"  Input shape: {max_tokens.shape}")
print(f"  Output shape: {max_embedded.shape}")

MAX Embedding:
  Input shape: [Dim(4)]
  Output shape: [Dim(4), Dim(128)]


### Custom Modules


In [28]:
# ============ PyTorch Custom Module ============
class PyTorchMLP(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(in_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        x = F_torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

pt_mlp = PyTorchMLP(784, 128, 10)

for name, param in pt_mlp.named_parameters():
    print(f"{name}: {param}")

pt_x = torch.randn(1, 784)
pt_out = pt_mlp(pt_x)
print(f"PyTorch MLP output: {pt_out[0, :10]}")

fc1.weight: Parameter containing:
tensor([[ 0.0198, -0.0146, -0.0313,  ...,  0.0228,  0.0063, -0.0089],
        [-0.0008,  0.0202, -0.0251,  ..., -0.0081,  0.0012, -0.0097],
        [-0.0195, -0.0215, -0.0018,  ...,  0.0145,  0.0271,  0.0010],
        ...,
        [-0.0174,  0.0119,  0.0198,  ..., -0.0032, -0.0244,  0.0119],
        [ 0.0327,  0.0174,  0.0213,  ..., -0.0085,  0.0306,  0.0247],
        [ 0.0342,  0.0022, -0.0302,  ...,  0.0246,  0.0122, -0.0306]],
       requires_grad=True)
fc1.bias: Parameter containing:
tensor([ 0.0148,  0.0227,  0.0100,  0.0304, -0.0057,  0.0202, -0.0121, -0.0333,
         0.0159,  0.0323,  0.0287,  0.0236,  0.0279, -0.0015, -0.0060, -0.0312,
        -0.0299, -0.0035, -0.0234,  0.0092,  0.0045, -0.0063, -0.0312,  0.0236,
         0.0322,  0.0188, -0.0224, -0.0261,  0.0319, -0.0157,  0.0205, -0.0297,
        -0.0079, -0.0294,  0.0294,  0.0008, -0.0156, -0.0346,  0.0328, -0.0032,
         0.0012, -0.0035,  0.0074,  0.0182,  0.0053, -0.0240,  0.0060, -0

In [29]:
# ============ MAX Custom Module ============
import max.nn.module_v3 as nn
from max.experimental import functional as F

class MAXMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        # NO need for: super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def __call__(self, x: Tensor) -> Tensor:
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

max_mlp = MAXMLP(784, 128, 10)

# TODO: print with ellipsis for compact repr
# for name, param in max_mlp.parameters:
#     print(f"{name}: {param}")

max_x = random.normal([1, 784])
max_out = max_mlp(max_x)
print(f"MAX MLP output shape: {max_out[0, :10]}")

MAX MLP output shape: TensorType(dtype=bfloat16, shape=[Dim(10)], device=gpu:0): [-292.0, -11.5625, 71.0, 500.0, 242.0, -672.0, -404.0, 294.0, -45.75, 332.0]


### Module System Comparison

| Concept | PyTorch | MAX |
|---------|---------|-----|
| Base class | `nn.Module` | `nn.Module` |
| Forward method | `def forward(self, x):` | `def __call__(self, x):` |
| Linear layer | `nn.Linear(in, out)` | `nn.Linear(in_dim, out_dim)` |
| Embedding | `nn.Embedding(vocab, dim)` | `nn.Embedding(vocab_size, dim)` |
| Parameters | `model.named_parameters()` | `model.parameters` (property) |
| Device transfer | `model.to('cuda')` | `model.to(Accelerator())` |


### Using @module_dataclass for Clean Definitions


In [30]:
import max.nn.module_v3 as nn

@nn.module_dataclass
class MLPDataclass(nn.Module):
    """Using @module_dataclass for automatic parameter tracking."""
    fc1: nn.Linear
    fc2: nn.Linear

    def __call__(self, x: Tensor) -> Tensor:
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Create with explicit layer instances
mlp = MLPDataclass(
    fc1=nn.Linear(784, 128),
    fc2=nn.Linear(128, 10)
)

print(f"Model: {mlp}")
print(f"\nParameters:")
for name, param in mlp.parameters:
    print(f"  {name}: {param.shape}")

Model: MLPDataclass(
    fc1=Linear(in_dim=Dim(784), out_dim=Dim(128)),
    fc2=Linear(in_dim=Dim(128), out_dim=Dim(10))
)

Parameters:
  fc1.weight: [Dim(128), Dim(784)]
  fc1.bias: [Dim(128)]
  fc2.weight: [Dim(10), Dim(128)]
  fc2.bias: [Dim(10)]


### Sequential


In [31]:
# ============ PyTorch Sequential ============
pt_seq = torch.nn.Sequential(
    torch.nn.Linear(784, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10)
)
print(f"PyTorch Sequential: {pt_seq}")

PyTorch Sequential: Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=10, bias=True)
)


In [32]:
# ============ MAX Sequential ============
from max.nn.module_v3 import Sequential

# Note: MAX Sequential expects unary modules (input -> output)
# For activations, we need to wrap them
@nn.module_dataclass
class ReLUModule(nn.Module):
    def __call__(self, x: Tensor) -> Tensor:
        return F.relu(x)

max_seq = Sequential(
    nn.Linear(784, 128),
    ReLUModule(),
    nn.Linear(128, 10)
)

sample_input = random.uniform([1, 784])

print(f"MAX Sequential: {max_seq}")
print(f"Sample output: {max_seq(sample_input)}")

MAX Sequential: Sequential(
    Linear(in_dim=Dim(784), out_dim=Dim(128)),
    ReLUModule(),
    Linear(in_dim=Dim(128), out_dim=Dim(10))
)
Sample output: TensorType(dtype=bfloat16, shape=[Dim(1), Dim(10)], device=gpu:0): [-83.5, -86.0, 111.5, -246.0, 138.0, -4.0625, -51.0, 398.0, -111.5, 24.625]


In [33]:
import max.nn.module_v3 as nn
from typing import Callable

@nn.module_dataclass
class Lambda(nn.Module):
    fn: Callable[..., Tensor]

    def __call__(self, x: Tensor) -> Tensor:
        return self.fn(x)

    def __rich_repr__(self):
        name = getattr(self.fn, "__name__", None) or repr(self.fn)
        yield name


max_seq = Sequential(
    nn.Linear(784, 128),
    Lambda(F.relu),
    nn.Linear(128, 10)
)
print(f"MAX Sequential: {max_seq}")
print(f"Sample output: {max_seq(sample_input)}")

MAX Sequential: Sequential(
    Linear(in_dim=Dim(784), out_dim=Dim(128)),
    Lambda('mo_relu'),
    Linear(in_dim=Dim(128), out_dim=Dim(10))
)
Sample output: TensorType(dtype=bfloat16, shape=[Dim(1), Dim(10)], device=gpu:0): [-39.0, 61.0, -121.0, -184.0, -135.0, 278.0, -120.5, 56.5, -16.25, 39.75]


---
## 5. Real Model: GPT-2 Attention in MAX (~5 min)

Let's translate a real model component from HuggingFace: **GPT-2 Multi-Head Attention**.

This demonstrates how to port production code from PyTorch to MAX.


In [34]:
# ============ PyTorch GPT-2 Attention (Simplified from HuggingFace) ============
# Reference: https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
import math

class PyTorchGPT2Attention(torch.nn.Module):
    """GPT-2 style multi-head attention."""

    def __init__(self, embed_dim=768, num_heads=12):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Combined QKV projection (like GPT-2)
        self.c_attn = torch.nn.Linear(embed_dim, 3 * embed_dim)
        # Output projection
        self.c_proj = torch.nn.Linear(embed_dim, embed_dim)

    def forward(self, hidden_states):
        batch, seq_len, _ = hidden_states.shape

        # Project to Q, K, V
        qkv = self.c_attn(hidden_states)
        q, k, v = qkv.split(self.embed_dim, dim=-1)

        # Reshape for multi-head: (batch, seq, heads, head_dim) -> (batch, heads, seq, head_dim)
        q = q.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        scale = 1.0 / math.sqrt(self.head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn_probs = F_torch.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_probs, v)

        # Reshape back: (batch, heads, seq, head_dim) -> (batch, seq, embed_dim)
        attn_output = attn_output.transpose(1, 2).reshape(batch, seq_len, self.embed_dim)

        # Output projection
        return self.c_proj(attn_output)

# Test PyTorch attention
pt_attn = PyTorchGPT2Attention(embed_dim=64, num_heads=4)
pt_hidden = torch.randn(2, 10, 64)  # batch=2, seq=10, dim=64
pt_result = pt_attn(pt_hidden)
print(f"PyTorch GPT-2 Attention output: {pt_result.shape}")

PyTorch GPT-2 Attention output: torch.Size([2, 10, 64])


In [36]:
# Translate MAX version live! 
import math

class MAXGPT2Attention(nn.Module):
    """GPT-2 style multi-head attention in MAX."""
    ...

# max_attn = MAXGPT2Attention(embed_dim=64, num_heads=4)
# max_hidden = random.uniform([2, 10, 64])  # batch=2, seq=10, dim=64
# max_result = max_attn(max_hidden)
# print(f"MAX GPT-2 Attention output: {max_result.shape}")

In [ ]:
# ============ Full GPT-2 Block in MAX ============
class MAXGPT2Block(nn.Module):
    """A complete GPT-2 transformer block."""

    def __init__(self, embed_dim=768, num_heads=12, ff_dim=None):
        super().__init__()
        ff_dim = ff_dim or 4 * embed_dim

        # Attention
        self.attn = MAXGPT2Attention(embed_dim, num_heads)

        # MLP (Feed-Forward Network)
        self.c_fc = nn.Linear(embed_dim, ff_dim)
        self.c_proj = nn.Linear(ff_dim, embed_dim)

        # Note: LayerNorm would go here in full implementation
        # MAX has LayerNorm in max.nn.norm

    def __call__(self, x: Tensor) -> Tensor:
        # Attention with residual
        attn_out = self.attn(x)
        x = x + attn_out

        # MLP with residual
        mlp_out = self.c_proj(F.gelu(self.c_fc(x)))
        x = x + mlp_out

        return x

# Test the full block
gpt2_block = MAXGPT2Block(embed_dim=64, num_heads=4)
x = random.uniform([2, 10, 64])
y = gpt2_block(x)
print(f"GPT-2 Block output: {y.shape}")

# Show all parameters
print("\nParameters:")
for name, param in gpt2_block.parameters:
    print(f"  {name}: {param.shape}")

---
## 6. Model Compilation for Optimized Inference (~2 min)

MAX can compile models into optimized graphs for faster inference.


In [37]:
# Create a simple model
model = MAXMLP(784, 128, 10)

# Create sample input
sample_input = random.uniform([1, 784])

# Run eager (interpreted) mode
eager_output = model(sample_input)
print(f"Eager output: {eager_output}")

# Compile the model for the input type
compiled_model = model.compile(sample_input.type)

# Run compiled (optimized graph) mode
compiled_output = compiled_model(sample_input)
print(f"Compiled output: {compiled_output}")

print(f"\nModel type: {type(model)}")
print(f"Compiled type: {type(compiled_model)}")

Eager output: TensorType(dtype=bfloat16, shape=[Dim(1), Dim(10)], device=gpu:0): [-127.5, -50.25, 19.25, 6.875, -18.5, 11.8125, 195.0, -62.75, -292.0, 72.0]
Compiled output: TensorType(dtype=bfloat16, shape=[Dim(1), Dim(10)], device=gpu:0): [-127.5, -50.25, 19.25, 6.875, -18.5, 11.8125, 195.0, -62.75, -292.0, 72.0]

Model type: <class '__main__.MAXMLP'>
Compiled type: <class 'function'>


In [38]:
# Quick benchmark: Eager vs Compiled model using Kepler
# pip install kepler
import time
import kepler
import torch

@kepler.measurement
def time_gpu():
    start_time = time.perf_counter_ns()
    start, end = torch.cuda.Event(True), torch.cuda.Event(True)
    start.record()
    yield
    end.record()
    torch.cuda.synchronize()
    return kepler.TimingEvent(start_time, start.elapsed_time(end) * 1e6)

with kepler.context.Context():
    with time_gpu("Eager Mode"):
        model(sample_input)._sync_realize()

    with time_gpu("Compiled Mode"):
        compiled_model(sample_input)

    kepler.report()

                                            Timings for  ⏱                                            
┏━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ Stage         ┃ Count ┃  Total ┃ Average ┃    Min ┃  Histogram ┃    Max ┃    P50 ┃    P90 ┃    P99 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ Eager Mode    │     1 │ 7.64 s │  7.64 s │ 7.64 s │ ⠀⠀⠀⠀⠀⡇⠀⠀⠀⠀ │ 7.64 s │ 7.64 s │ 7.64 s │ 7.64 s │
│ Compiled Mode │     1 │ 438 μs │  438 μs │ 438 μs │ ⠀⠀⠀⠀⠀⡇⠀⠀⠀⠀ │ 438 μs │ 438 μs │ 438 μs │ 438 μs │
└───────────────┴───────┴────────┴─────────┴────────┴────────────┴────────┴────────┴────────┴────────┘

---
## Summary: PyTorch to MAX Cheat Sheet

### Imports
```python
# PyTorch                           # MAX
import torch.nn as nn               import max.nn.module_v3 as nn
import torch.nn.functional as F     from max.experimental import Tensor, functional as F
                                    from max.driver import CPU, Accelerator
                                    from max.dtype import DType
```

### Tensors
```python
# PyTorch                           # MAX
torch.tensor([1,2,3])               Tensor.constant([1,2,3])
torch.zeros(3, 4)                   Tensor.zeros([3, 4])
torch.randn(2, 3)                   random.normal([2, 3])
x.to('cuda')                        x.to(Accelerator())
x.float()                           x.cast(DType.float32)
```

### Modules
```python
# PyTorch                           # MAX
class Model(nn.Module):             class Model(nn.Module):
    def forward(self, x):               def __call__(self, x):
        return x                            return x

nn.Linear(in, out)                  nn.Linear(in_dim, out_dim)
nn.Embedding(vocab, dim)            nn.Embedding(vocab_size, dim)
```

### Operations
```python
# PyTorch                           # MAX
F.softmax(x, dim=-1)                F.softmax(x)
x.sum(dim=1)                        F.sum(x, axis=1)
x.view(2, 3)                        x.reshape(2, 3) == F.reshape(x, [2, 3])
x.transpose(0, 1)                   x.T == x.transpose(0, 1) == F.transpose(x, 0, 1)
```

### Inference
```python
# PyTorch                           # MAX
model.eval()                        compiled = model.compile(input.type)
torch.compile(model)                output = compiled(input)
with torch.no_grad():               
    output = (input)
```


---
## Next Steps

Now that you understand the basics of MAX's modeling API:

1. **Explore more layers**: Check `max.nn` for attention, normalization, and more
2. **Load pretrained weights**: Use `model.load_state_dict()` to load weights
3. **Optimize for production**: Explore MAX's graph compilation features
4. **Run larger models**: MAX is optimized for LLM inference at scale

### Resources
- [MAX Experimental API](https://docs.modular.com/max/api/python/experimental/)
- [MAX Module V3](https://docs.modular.com/max/api/python/nn/module_v3)
- [MAX Documentation](https://docs.modular.com/max/)
- [LLM Book in MAX](https://llm.modular.com/)